## Preprocessing code

In this notebook ill hold all code to do with preprocessing done to the image data to generate my dataset. This will involve:
* Splitting each sample inot a folder of frames
* Extracting pose data from each of those frames
* Experimenting with cleaned and noprmalised data
* Sanity testing extraction pipeline

In [1]:
import os
import json
import shutil
import datetime


import cv2 as cv
import numpy as np
import pandas as pd
import seaborn as sns
# import mediapipe as mp
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib.ticker import ScalarFormatter
sns.set_theme(style="whitegrid", font_scale=1.2) 

# Class Imports
# from cleaning.Normalizer import *
from classes.Annotation import Annotater
from classes.GraphPlotting import GraphPlotter
from classes.KeypointExtractor import KeypointExtractor


Utility functions that I use throughout this script

In [2]:
def pretty_print_json(dict_object: object):
    ''' Makes JSON object prettier for being printed 
    '''
    json_object = json.dumps(dict_object)
    parsed_json = json.loads(json_object) 
    print(json.dumps(parsed_json, indent=4))


def save_JSON_object(json_path: str, json_object: object):
    ''' Saves a JSON Object to a path
    '''
    with open(json_path, 'w') as outfile:
        json.dump(json_object, outfile)


def load_JSON_object(json_path: str):
    ''' Loads JSON object from a file
    '''
    with open(json_path, "r") as f:
        json_data = json.load(f)
        return json_data


def create_folder(folder_name: str):
    '''Used to create folders
    '''
    if not os.path.exists(folder_name):
        os.mkdir(folder_name)
    

def build_all_folders():
    ''' Helper function for building all neccesary folders for extracting JSON data
    '''
    create_folder('data/')
    create_folder('data/video_frames/')
    create_folder('data/json_data/')


def get_total_action_counts(json_folders):
    ''' Get total action count of all actions in dataset
        Replace passing args with list of file paths for when I added synthesized subsampled data
    Args:
        json_folder_path () :
        mirrored_json_folder_path () :
    '''
    action_set = {'Backhand' : 0, 'Forehand' : 0, 'Serves' : 0, 'NoStroke' : 0}
    print(action_set)
    for json_folder in json_folders:
        for action in action_set:
            print(f'FOLDER : {json_folder} : ACTION: {action}, LEN : {len(os.listdir(f"{json_folder}{action}"))}...')


## Notes

Links to online datasets:
    
* Kaggle swing dataset : https://www.kaggle.com/datasets/orvile/tennis-player-actions-dataset?resource=download
* Carlos Alcaraz up close practice : https://www.youtube.com/watch?v=MFpSeZRx83g

Dataset compriised of:
* Backhand
* Forehand
* Serve
* Ready Position (Idle)

This comes with an images and annotations folder which I will use to stick series of images back toghet to mkae videdos so that I can extract and collate JSON data easier

## Refactoring plan for this

Works fine but convoluted can just us keypointExtractor for all files and add a mirror flag to create a mirrored file at the same time with all x inverted

In [3]:
def examine_annotations(folder_path):
    ''' See if i can use the annotations to stitch the images back together 
    Args:
    '''
    for swing_anno_file in os.listdir(folder_path):
        anno_data = load_JSON_object(f'{folder_path}{swing_anno_file}')

        frame_collector = {}
        # pretty_print_json(anno_data)
        # Get unique_dataset ID's
        ids = set([anno['file_name'] for anno in anno_data['images']])

        for anno in anno_data['images']:
            if anno['file_name'] not in frame_collector.keys():
            #    frame_collector[anno['file_name']] =
                wont_work= 10 
        # NOTE - Redo this and just flip images & do 1 - all x-coords
        

# examine_annotations('data/kaggle_dataset/annotations/')

In [4]:
def extract_pose_data(dataset_folder: str, output_frame_path:str, output_JSON_path:str):
    ''' Iterates thorugh the video dataset folder and create a folder of pose JSON
    Args:
        dataset_folder (String) : String file path to fodler containing Guenoles dataset of tennis swings
    '''

    for swing_type in os.listdir(dataset_folder):
        # Extratc JSON from all video files
        for file in tqdm(os.listdir(f'{dataset_folder}{swing_type}/')):
            # print(f'Extracting : {file}')
            # create_folder(f'data/video_frames/{swing_type}/')
            # create_folder(f'data/json_data/{swing_type}/')
            
            extractor = KeypointExtractor(video_frame_path=f'{output_frame_path}/{swing_type}/', 
                                          output_json_path=f'{output_JSON_path}/{swing_type}/', 
                                          method='norm',)
            extractor.video_to_json(f'{dataset_folder}{swing_type}/{file}')
        print("FINISHED")

In [5]:
def horizontally_flip_dataset(video_frame_folder: str, output_path:str ='data/videos/mirrored_videos/'):
    ''' Funtction for flipping dataset and updating resulting action
    Links:
        How to flip image : https://www.opencvhelp.org/tutorials/image-processing/how-to-flip-image/
        How to write to a file correctly : https://stackoverflow.com/questions/6159900/correct-way-to-write-line-to-file
    Args:
        dataset_path (String) : String path to folder whose contents we want to flip horizontally
        output_path (String) : File path to save folder of horizontally flipped images to
    '''
    # Iterate through dataset and horizontally flip all images to double dataset size
    create_folder(output_path)
    for swing_type in os.listdir(video_frame_folder):
        # Create folders to store mirrored frames & videos
        create_folder(f'{output_path}{swing_type}/')
        create_folder(f'data/videos/mirrored_videos/{swing_type}/')
        for uid_folder in tqdm(os.listdir(f'{video_frame_folder}{swing_type}'), desc=swing_type):
            dataset_path = f'{video_frame_folder}{swing_type}/{uid_folder}/'
            create_folder(f'{output_path}{swing_type}/{uid_folder}/')
            for file_name in os.listdir(dataset_path):
                # Flip and save image
                image = cv.imread(f'{dataset_path}{file_name}')
                flipped_image = cv.flip(image, 1)
                # print(f'{dataset_path}{file_name}')
                # print(f'{output_path}{swing_type}/{uid_folder}/Mirrored_{file_name}')
                # a-b
                cv.imwrite(f'{output_path}{swing_type}/{uid_folder}/Mirrored_{file_name}', flipped_image)
            
            print(f'{output_path}{swing_type}/{uid_folder}/')
            print(f'data/videos/mirrored_videos/{swing_type}/')
            print(f'mirrored_{uid_folder}.mp4')
            # a-b
            annotater = Annotater([])
            annotater.output_frames_to_video(frame_folder_path=f'{output_path}{swing_type}/{uid_folder}/',
                                             output_video_path = f'data/videos/mirrored_videos/{swing_type}/',
                                             output_video_name = f'mirrored_{uid_folder}.mp4',
                                             fps=30, range=None)
        

In [6]:
def plot_all_annotations(json_folder_path, frame_folder_path, output_folder_base=None):
    ''' Folder to plot all skeletal data over the annotations
    Args:
        json_folder_path (String) : String file path to folder of all swing JSON
        frame_folder_path (String) : String file path to folder of all video frames
        output_folder_base (String) : String file path to base folder to store annoatted folders of frames
    '''
    create_folder(output_folder_base)
    for swing_type in os.listdir(json_folder_path):
        # Create folder to store all annotated folders of frames for that wsing type
        create_folder(f'{output_folder_base}{swing_type}/')
        
        for file in tqdm(os.listdir(f'{json_folder_path}{swing_type}/')):
            # Load JSON data
            json_data = load_JSON_object(f'{json_folder_path}{swing_type}/{file}')
            file_frame_folder_path = f'{frame_folder_path}{swing_type}/{file.split(".")[0]}/'
            output_folder_path = f'{output_folder_base}{swing_type}/{file.split(".")[0]}/'

            # Get image dimensions
            file_list = os.listdir(file_frame_folder_path)
            img = cv.imread(f'{file_frame_folder_path}{file_list[0]}')
            height, width, _ = img.shape


            create_folder(output_folder_path)
            # REMOVE AT SOME POINT
            # dm = Normalizer(json_data, _, _, _, _, age_lower, fps, width, height)

            # print('json_folder_path:', json_folder_path)
            # print('frame_folder_path:', frame_folder_path)
            # print('output_folder_base:', output_folder_base)
            # a-b
            # annotater = Annotater(dm.cleaned_df)
            # annotater.annotate_frames(file_frame_folder_path, 
            #                   dm.cleaned_df,
            #                   output_folder_path,
            #                   ratios=[width, height])


In [7]:
def create_mirrored_videos(frame_path, output_path):
    ''' Stitch folders of mirrored frames back together 
    Links:
        stitch frames : https://stackoverflow.com/questions/43048725/python-creating-video-from-images-using-opencv
    Args:
        frame_path (String) : 
        output_path (String) :
    '''
    for swing_type in os.listdir(frame_path):
        create_folder(f'{output_path}{swing_type}/')
        for frame_folder in tqdm(os.listdir(f'{frame_path}{swing_type}/'), desc=swing_type, leave=False):
            # Sort frames numerically
            frame_nums = {int(file_name.split(".")[0].replace("Mirrored_", "")) : file_name for file_name in os.listdir(f'{frame_path}{swing_type}/{frame_folder}/')}
            sorted_frames = sorted(frame_nums.keys())

            # Get image dimensions
            img = cv.imread(f'{frame_path}{swing_type}/{frame_folder}/{frame_nums[0]}')
            height, width, _ = img.shape

            # write frames back to video
            fourcc = cv.VideoWriter_fourcc(*'mp4v') 
            video = cv.VideoWriter(f'{output_path}{swing_type}/Mirrored_{frame_folder}.mp4', fourcc, 30, (width, height))
            for frame_idx in sorted_frames:
                img = cv.imread(f'{frame_path}{swing_type}/{frame_folder}/{frame_nums[frame_idx]}')
                video.write(img)

            cv.destroyAllWindows()
            video.release()

## Main cell to run functions above

In [8]:
# build_all_folders()

video_folder_path = '../VideoDataset/'
# extract_pose_data(video_folder_path)

# Plot skeleton annotations
json_folder_path = 'data/json_data/raw/'
frame_folder_path = 'data/video_frames/raw/'
annotated_frame_folder = 'data/video_frames/annotated_raw_frames/'

mirrored_json_folder_path = 'data/json_data/mirrored_json_data/'
mirrored_video_folder_path = 'data/videos/mirrored_videos/'
mirrored_frame_folder_path = 'data/video_frames/mirrored_video_frames/'
mirrored_annotated_frame_folder = 'data/video_frames/mirrored_annotated_frames/'

# NOTE - Extract pose data on flipped videos
# create_folder('data/video_frames/')
# create_folder('data/json_data/')
# create_folder('data/json_data/raw/')
# create_folder('data/video_frames/raw/')
# extract_pose_data('data/videos/raw/', 'data/video_frames/raw/', 'data/json_data/raw/')


# NOTE - Horizontally flip dataset
# Horizontally flip dataset
horizontally_flip_dataset(frame_folder_path, output_path='data/video_frames/mirrored_video_frames/')

# Stictch frames back together to make mirrored videos
create_mirrored_videos(mirrored_frame_folder_path, mirrored_video_folder_path)


extract_pose_data('data/videos/mirrored_videos/', 'data/mirrored_video_frames/', 'data/mirrored_json_data/')

# NOTE - Plot annotations back over images as sanity test
# plot_all_annotations(json_folder_path, frame_folder_path, output_folder_base=annotated_frame_folder)
# plot_all_annotations(mirrored_json_folder_path, mirrored_frame_folder_path, output_folder_base=mirrored_annotated_frame_folder)

# Get total action instance count in dataset
json_folders = [json_folder_path, mirrored_json_folder_path]
get_total_action_counts(json_folders)

Backhand:   0%|          | 0/20 [00:00<?, ?it/s]

data/video_frames/mirrored_video_frames/Backhand/BH_1/
data/videos/mirrored_videos/Backhand/
mirrored_BH_1.mp4


Backhand:   5%|▌         | 1/20 [00:02<00:51,  2.72s/it]

An error occurred: 'data/videos/mirrored_videos/Backhand/' is a directory
data/video_frames/mirrored_video_frames/Backhand/BH_10/
data/videos/mirrored_videos/Backhand/
mirrored_BH_10.mp4


Backhand:  10%|█         | 2/20 [00:05<00:52,  2.91s/it]

An error occurred: 'data/videos/mirrored_videos/Backhand/' is a directory
data/video_frames/mirrored_video_frames/Backhand/BH_11/
data/videos/mirrored_videos/Backhand/
mirrored_BH_11.mp4


Backhand:  15%|█▌        | 3/20 [00:08<00:51,  3.00s/it]

An error occurred: 'data/videos/mirrored_videos/Backhand/' is a directory


Backhand:  15%|█▌        | 3/20 [00:09<00:52,  3.07s/it]


KeyboardInterrupt: 

## Merge Raw & Mirrored datasets 

In [19]:
base = 'data/json_data/'
create_folder(f'{base}merged/')
for swing_type in os.listdir(f'{base}raw/'):
    for file in tqdm(os.listdir(f'{base}raw/{swing_type}/')):
        shutil.copy(f'{base}raw/{swing_type}/{file}', f'{base}merged/{file}')
        shutil.copy(f'{base}mirrored_json_data/{swing_type}/Mirrored_{file}', f'{base}merged/Mirrored_{file}')

100%|██████████| 20/20 [00:00<00:00, 39.11it/s]


## Exploratory Data Analysis

In this section ill be looking at the following:
* Num videos per action type
* FPS Distribution for potential frame subsampling
* Mirroring dataset to double size
* Anything else that comes to mind later


In [ ]:
''' 
Links:
    video duration : https://www.geeksforgeeks.org/python/get-video-duration-using-python-opencv/
    histpolot : https://seaborn.pydata.org/generated/seaborn.histplot.html
    title fix : https://seaborn.pydata.org/generated/seaborn.despine.html
'''

def plot_val_distribution(fps_dict:dict):
    '''Plot a clean, professional histogram showing FPS distribution
    '''
    fps_values = list(fps_dict.values())
    plt.figure(figsize=(8, 5))
    sns.histplot(fps_values, bins=len(set(fps_values)), color="#3A7CA5", edgecolor='white', kde=True, alpha=0.8)

    mean_val = np.mean(fps_values)
    plt.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean = {mean_val:.2f}')

    plt.xlabel("Frames per Second (FPS)", fontsize=13, labelpad=10)
    plt.ylabel("Number of Videos", fontsize=13, labelpad=10)
    plt.title("Distribution of Video Frame Rates", fontsize=15, pad=15)
    plt.legend()
    plt.tight_layout()
    plt.show()



def plot_hist_distribution(size_dict:dict, title:str):
    ''' Plot a clean, professional bar chart comparing class sizes 
    Args:
        size_dict (Dict) : Dict containing  sizes for each swing type 
        title (String) : Title of plot
    '''
    labels = list(size_dict.keys())
    values = np.array(list(size_dict.values()))

    # Sort bars by value for clearer comparison
    sorted_idx = np.argsort(labels)[::-1]
    labels = np.array(labels)[sorted_idx]
    values = values[sorted_idx]
    palette = sns.color_palette("crest", n_colors=len(labels))
    plt.figure(figsize=(8, 5))
    bars = plt.bar(labels, values, color=palette, edgecolor="black", alpha=0.9)

    # Add annotations (counts)
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, val + 0.5, f'{val}', ha='center', va='bottom', fontsize=11, fontweight='medium')

    plt.xlabel("Action Class", fontsize=13, labelpad=10)
    plt.ylabel("Number of Samples", fontsize=13, labelpad=10)
    plt.title(title, fontsize=15, pad=15)
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()



def plot_duration_candles(durations_dict:dict, show_violin:bool =False):
    ''' Plot duration distributions per action (as boxplots or violins).
        durations_dict = {
            "Forehand": {"vid1.mp4": 2.4, "vid2.mp4": 2.1},
            "Backhand": {"vid3.mp4": 3.0, "vid4.mp4": 2.7},
        }
    '''
    records = []
    for action, files in durations_dict.items():
        for filename, duration in files.items():
            records.append({"Action": action, "File": filename, "Duration": duration})
    df = pd.DataFrame(records, columns=["Action", "File", "Duration"])

    # Sort by median duration
    order = df.groupby("Action")["Duration"].median().sort_values(ascending=False).index

    # Plot candles of durations
    plt.figure(figsize=(9, 5))
    plot_fn = sns.violinplot if show_violin else sns.boxplot
    plot_fn(data=df, x="Action", y="Duration", order=order, palette="crest", showmeans=not show_violin, meanline=True, meanprops={"color": "red", "ls": "--", "lw": 1.5})
    plt.title("Distribution of Action Durations", fontsize=15, weight="bold")
    plt.xlabel("Action Type")
    plt.ylabel("Duration (seconds)")
    plt.xticks(rotation=15, ha="right")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()



def plot_dimension_distributions(size_dict):
    ''' Plot a clean, professional bar chart comparing class sizes 
    Args:
        size_dict (Dict) : Dict containing  sizes for each swing type 
    '''
    widths = [dims[0] for file_name, dims in size_dict.items()]
    heights = [dims[1] for file_name, dims in size_dict.items()]
    plt.figure(figsize=(12, 5))

    fig, axes = plt.subplots(1, 3, figsize=(12, 5))
    # plot widths
    axes[0].hist(widths, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_title('Distribution of Image Widths')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Count')
    axes[0].xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    axes[0].ticklabel_format(style='plain', axis='x')

    # Plot heights
    axes[1].hist(heights, bins=20, color='seagreen', edgecolor='black', alpha=0.7)
    axes[1].set_title('Distribution of Image Heights')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Count')
    axes[1].xaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    axes[1].ticklabel_format(style='plain', axis='x')

    axes[2].scatter(widths, heights, alpha=0.6, edgecolor='k')
    axes[2].set_xlabel("Width (pixels)")
    axes[2].set_ylabel("Height (pixels)")
    axes[2].set_title("Image dims scatter")
    plt.show()

    plt.tight_layout()
    plt.show()
    

    


# Get distribution of FPS's in dataset
dimensions_dict = {}
fps_dict = {}
size_dict = {}
frame_duration = {'Forehand': [], 'Backhand': [], 'Serves': [], 'NoStroke': []}
durations_dict = {}
for swing_type in os.listdir('data/videos/raw/'):
    file_list = os.listdir(f'data/videos/raw/{swing_type}')
    size_dict[swing_type] = len(file_list)
    durations_dict[swing_type] = {}
    
    for file_name in tqdm(file_list):
        # Extract FPS & duration of each video
        vidcap = cv.VideoCapture(f'data/videos/raw/{swing_type}/{file_name}')
        frames = vidcap.get(cv.CAP_PROP_FRAME_COUNT)
        fps = vidcap.get(cv.CAP_PROP_FPS)
        # Store metadata
        fps_dict[file_name] = int(fps)
        frame_duration[swing_type].append(frames)
        durations_dict[swing_type][file_name] = frames 
        # Store width adn height
        video_dims = (int(vidcap.get(3)), int(vidcap.get(4)))
        dimensions_dict[file_name] = video_dims

# pretty_print_json(durations_dict)
# a-b

action_frame_avgs = {k : sum(v) / len(v) for k, v in frame_duration.items()}
print(action_frame_avgs)
plot_val_distribution(fps_dict)
plot_hist_distribution(size_dict, title='Distribution of Action Samples per Class')
plot_hist_distribution(action_frame_avgs, title='Distribution of Video Lengths per Class (Frames)')
plot_duration_candles(durations_dict)
plot_dimension_distributions(dimensions_dict)

gp = GraphPlotter('')
gp.plot_side_by_side('data/video_frames/annotated_raw_frames/Forehand/FH_8/11.jpg', 'data/video_frames/mirrored_annotated_frames/Forehand/Mirrored_FH_8/11.jpg', 'FH_8 (Original)', 'FH_8 (Flipped)')